# Discovery Data Mentah 



In [ ]:
from pathlib import Path
import re, json, unicodedata, warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

BASE_DIR = Path.cwd()

SEARCH_DIRS = [BASE_DIR, BASE_DIR / "project_raw", BASE_DIR.parent, BASE_DIR.parent / "project_raw"]

TAHUN_LIST  = [2023, 2024, 2025]
N_KELURAHAN = 153
BBOX_LAT    = (-7.45, -7.10)     # batas pemeriksaan Surabaya
BBOX_LON    = (112.55, 112.90)

KECAMATAN_RESMI = [
    "Asemrowo", "Benowo", "Bubutan", "Bulak", "Dukuh Pakis", "Gayungan", "Genteng", "Gubeng",
    "Gunung Anyar", "Jambangan", "Karang Pilang", "Kenjeran", "Krembangan", "Lakarsantri",
    "Mulyorejo", "Pabean Cantian", "Pakal", "Rungkut", "Sambikerep", "Sawahan", "Semampir",
    "Simokerto", "Sukolilo", "Sukomanunggal", "Tambaksari", "Tandes", "Tegalsari",
    "Tenggilis Mejoyo", "Wiyung", "Wonocolo", "Wonokromo",
]

def find_file(patterns, deskripsi):
    dicari = []
    for d in SEARCH_DIRS:
        for pat in patterns:
            hits = sorted(p for p in d.glob(pat) if not p.name.startswith("~$"))
            if hits:
                return hits[0]
        dicari.append(str(d))
    raise FileNotFoundError(f"File {deskripsi} tidak ditemukan. Folder yang dicari: {dicari}. "
                            f"Pola: {patterns}")

FILES = {
    y: find_file([f"mediacenter.keluhan (Tahun {y}).xlsx", f"*keluhan*{y}*.xlsx"], f"pengaduan {y}")
    for y in TAHUN_LIST
}
COORD_FILE = find_file(
    ["koordinat_kelurahan_2023_2025.csv", "koordinat_kelurahan*2023_2025*.csv", "koordinat_kelurahan*.csv"],
    "koordinat kelurahan",
)


def normalize_location(value):
    if pd.isna(value):
        return pd.NA
    v = unicodedata.normalize("NFKC", str(value)).casefold()
    v = re.sub(r"\s+", " ", v).strip()
    return v if v else pd.NA

def kec_key(value):
    # kunci kecamatan tanpa spasi: 'Gunung Anyar' == 'Gununganyar'
    n = normalize_location(value)
    return pd.NA if pd.isna(n) else n.replace(" ", "")

KEC_RESMI_KEY = {kec_key(k) for k in KECAMATAN_RESMI}
PHONE_RE = re.compile(r"(?<!\d)(?:\+?62|0)8\d{8,11}(?!\d)")

print("Folder kerja :", BASE_DIR)
for y, p in FILES.items():
    print(f"Pengaduan {y}:", p)
print("Koordinat    :", COORD_FILE)

## 1. Load data mentah 

In [ ]:
raw = {y: pd.read_excel(p) for y, p in FILES.items()}
coords = pd.read_csv(COORD_FILE)

for y, d in raw.items():
    print(f"{y}: {d.shape[0]:,} baris × {d.shape[1]} kolom")
print(f"Koordinat: {coords.shape[0]:,} baris × {coords.shape[1]} kolom")

# Gabungan HANYA untuk pemeriksaan lintas tahun (raw per tahun tidak diubah)
allr = pd.concat([d.assign(Tahun_file=y) for y, d in raw.items()], ignore_index=True)
print("Total gabungan:", f"{len(allr):,}")

## 2. Struktur kolom

In [ ]:
kolom_per_tahun = {y: list(d.columns) for y, d in raw.items()}
sama = len({tuple(v) for v in kolom_per_tahun.values()}) == 1
print("Struktur kolom identik di semua tahun:", sama)
if not sama:
    for y, v in kolom_per_tahun.items():
        print(y, v)

WAJIB = ["No Ticket", "Instansi", "Topik", "Kategori", "Channel", "Keluhan", "Kelurahan",
         "Kecamatan", "Tanggal Keluhan"]
hilang = {y: [c for c in WAJIB if c not in d.columns] for y, d in raw.items()}
print("Kolom wajib yang hilang:", hilang)

d0 = raw[TAHUN_LIST[0]]
display(pd.DataFrame({"kolom": d0.columns, "dtype": [str(t) for t in d0.dtypes]}))


## 3. Missing values

In [ ]:
miss = pd.concat({
    y: pd.DataFrame({"missing": d.isna().sum(), "pct": (d.isna().mean() * 100).round(3)})
    for y, d in raw.items()
}, axis=1)
display(miss[(miss.xs("missing", axis=1, level=1) > 0).any(axis=1)])
print("Hanya kolom yang punya missing yang ditampilkan.")

## 4. Tiket & duplikasi
Tiga hal berbeda diperiksa: duplikat baris utuh, duplikat tiket **dalam** tahun, dan duplikat tiket **antar** tahun.

In [ ]:
n_dup_baris = int(allr.drop(columns="Tahun_file").duplicated().sum())
n_dup_tiket_dalam = int(allr.duplicated(["Tahun_file", "No Ticket"]).sum())
n_dup_tiket_total = int(allr["No Ticket"].duplicated().sum())
n_dup_tiket_antar = n_dup_tiket_total - n_dup_tiket_dalam

print("Duplikat baris utuh          :", n_dup_baris)
print("Duplikat tiket dalam 1 tahun :", n_dup_tiket_dalam)
print("Duplikat tiket antar tahun   :", n_dup_tiket_antar)

# Tahun pada nomor tiket (…/MC/XII/2023) vs tahun file: bukan error, hanya info
allr["tiket_tahun"] = pd.to_numeric(allr["No Ticket"].astype(str).str.extract(r"/(\d{4})$")[0], errors="coerce")
sel = allr[allr["tiket_tahun"].notna() & (allr["tiket_tahun"] != allr["Tahun_file"])]
print("Tiket yang tahun nomornya != tahun file (info):", len(sel))
display(sel[["No Ticket", "Tahun_file", "Tanggal Keluhan"]].head(10))

## 5. Validasi tanggal & konsistensi tahun

In [ ]:
allr["tgl"] = pd.to_datetime(allr["Tanggal Keluhan"], errors="coerce")
n_tgl_invalid = int(allr["tgl"].isna().sum())
print("Tanggal invalid:", n_tgl_invalid)

display(allr.groupby("Tahun_file")["tgl"].agg(["min", "max", "count"]))
print("\nTahun pada tanggal keluhan (baris) vs tahun file:")
ct = pd.crosstab(allr["Tahun_file"], allr["tgl"].dt.year)
display(ct)
n_tahun_beda = int((allr["tgl"].dt.year != allr["Tahun_file"]).sum())
print("Baris yang tahun tanggalnya != tahun file:", n_tahun_beda)

## 6. Lokasi mentah
Pemeriksaan: missing Kelurahan (dan polanya), kecamatan di luar 31 resmi, dan **pasangan Kelurahan–Kecamatan yang tidak konsisten**.

In [ ]:
allr["kel_norm"] = allr["Kelurahan"].map(normalize_location)
allr["kec_norm"] = allr["Kecamatan"].map(normalize_location)
allr["kec_k"]    = allr["Kecamatan"].map(kec_key)

n_kel_missing = int(allr["kel_norm"].isna().sum())
print("Kelurahan missing:", n_kel_missing, f"({n_kel_missing/len(allr)*100:.2f}%)")
print("Kecamatan missing:", int(allr["kec_norm"].isna().sum()))
print("Kelurahan missing tapi Kecamatan ada:", int((allr["kel_norm"].isna() & allr["kec_norm"].notna()).sum()))

print("\nMissing Kelurahan per Kecamatan × Tahun (apakah acak?):")
mk = allr[allr["kel_norm"].isna()]
tab_mk = pd.crosstab(mk["Kecamatan"], mk["Tahun_file"], margins=True)
display(tab_mk.sort_values("All", ascending=False))

# Kecamatan di luar daftar resmi
kec_tidak_resmi = sorted(set(allr.loc[allr["kec_k"].notna() & ~allr["kec_k"].isin(KEC_RESMI_KEY), "Kecamatan"].unique()))
print("\nKecamatan di luar 31 resmi (setelah abaikan spasi/huruf besar):", kec_tidak_resmi)
print("Kecamatan unik (ejaan mentah):", allr["Kecamatan"].nunique(), "| kelurahan unik:", allr["kel_norm"].nunique())

# Pasangan Kelurahan-Kecamatan
pair = (allr.dropna(subset=["kel_norm", "kec_norm"])
        .groupby(["Kelurahan", "Kecamatan"]).size().reset_index(name="n_baris"))
jml_kec = pair.groupby("Kelurahan")["Kecamatan"].nunique()
ambigu = pair[pair["Kelurahan"].isin(jml_kec[jml_kec > 1].index)].sort_values(["Kelurahan", "n_baris"], ascending=[True, False])
print("\nKelurahan yang muncul dengan >1 Kecamatan:", ambigu["Kelurahan"].nunique())
display(ambigu)

# baris di pasangan 'minoritas' (kemungkinan salah input kecamatan)
n_pair_minoritas = int(ambigu.groupby("Kelurahan")["n_baris"].apply(lambda s: s.sum() - s.max()).sum())
print("Baris pada pasangan minoritas (kemungkinan salah input Kecamatan):", n_pair_minoritas)

## 7. Teks `Keluhan`

In [ ]:
teks = allr["Keluhan"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
L = teks.str.len()

n_keluhan_missing = int(allr["Keluhan"].isna().sum())
n_keluhan_kosong  = int((teks == "").sum())
n_pendek = int((L <= 10).sum()); n_panjang = int((L >= 2000).sum())
print("Keluhan missing:", n_keluhan_missing, "| kosong setelah strip:", n_keluhan_kosong)
print("Panjang (karakter):"); display(L.describe().round(1).to_frame().T)
print(f"Teks <=10 karakter: {n_pendek} | >=2000 karakter: {n_panjang}")

print("\nContoh teks pendek (lihat apakah masih bermakna):")
display(teks[L <= 10].str.lower().value_counts().head(15).rename("n").to_frame())

# Teks identik berulang: sama persis + kelurahan sama + tahun sama
kunci = pd.DataFrame({"t": teks.str.casefold(), "k": allr["kel_norm"].astype("string"), "y": allr["Tahun_file"]})
valid_k = kunci["t"].notna() & (kunci["t"] != "") & kunci["k"].notna()
dup_mask = valid_k & kunci.duplicated(keep=False)
n_dup_teks_lok = int(dup_mask.sum())
print("\nBaris dengan teks identik di kelurahan & tahun yang sama:", n_dup_teks_lok)
rep = (kunci[dup_mask].groupby(["t", "k", "y"]).size().reset_index(name="n")
       .sort_values("n", ascending=False).head(10))
rep["t"] = rep["t"].str.slice(0, 100)
display(rep)

n_telepon = int(teks.fillna("").str.contains(PHONE_RE).sum())
print("Baris yang memuat pola nomor telepon:", n_telepon, "(data pribadi; jangan dikutip apa adanya di laporan)")

## 8. Kolom kategorikal

In [ ]:
for col in ["Kategori", "Channel", "Status"]:
    print("\n" + "=" * 70); print(col)
    display(allr[col].value_counts(dropna=False).rename("n").to_frame().T if allr[col].nunique() <= 8
            else allr[col].value_counts(dropna=False).head(10).rename("n").to_frame())
print("\nTopik unik:", allr["Topik"].nunique(), "| Instansi unik:", allr["Instansi"].nunique())
display(allr["Topik"].value_counts().head(15).rename("n").to_frame())

## 9. Kolom waktu (tidak dipakai pada penelitian ini; hanya dicatat)

In [ ]:
for c in ["Waktu Respon Tanggapan", "Waktu Tindak Lanjut", "Waktu Penyelesaian Pengaduan"]:
    if c in allr.columns:
        s = pd.to_numeric(allr[c], errors="coerce")
        print(f"{c:35s} valid={int(s.notna().sum()):6d}  negatif={int((s < 0).sum()):4d}  min={s.min()}  median={s.median()}  max={s.max()}")

## 10. Audit file koordinat
File ini **bukan data mentah murni** (dibuat pada tahap wrangling sebelumnya), sehingga asal-usul dan kualitasnya diaudit di sini.

In [ ]:
print("Kolom:", coords.columns.tolist())
WAJIB_K = ["kelurahan", "kecamatan", "lat_kelurahan", "lon_kelurahan"]
hilang_k = [c for c in WAJIB_K if c not in coords.columns]
if hilang_k:
    raise ValueError(f"Kolom koordinat wajib tidak ada: {hilang_k}")
has_src = "sumber_koordinat" in coords.columns
if not has_src:
    print("PERINGATAN: kolom sumber_koordinat tidak ada -> asal koordinat tidak bisa diaudit.")

c = coords.copy()
c["kel_norm"] = c["kelurahan"].map(normalize_location)
c["lat"] = pd.to_numeric(c["lat_kelurahan"], errors="coerce")
c["lon"] = pd.to_numeric(c["lon_kelurahan"], errors="coerce")

coord_checks = {
    "jumlah_baris": len(c),
    "kelurahan_unik": int(c["kel_norm"].nunique()),
    "nama_kelurahan_duplikat": int(c["kel_norm"].duplicated().sum()),
    "koordinat_kosong_atau_non_numerik": int(c[["lat", "lon"]].isna().any(axis=1).sum()),
    "koordinat_di_luar_bbox_surabaya": int((~c["lat"].between(*BBOX_LAT) | ~c["lon"].between(*BBOX_LON)).sum()),
    "pasangan_lat_lon_duplikat": int(c.duplicated(["lat", "lon"]).sum()),
}
if has_src:
    print("\nsumber_koordinat:"); display(c["sumber_koordinat"].value_counts(dropna=False).rename("n").to_frame())
    coord_checks["baris_fallback"] = int(c["sumber_koordinat"].astype(str).str.contains("fallback", case=False).sum())
display(pd.Series(coord_checks, name="nilai").to_frame())

kel_raw  = set(allr["kel_norm"].dropna().unique())
kel_koor = set(c["kel_norm"].dropna().unique())
print("Kelurahan ada di pengaduan tapi tidak di koordinat:", sorted(kel_raw - kel_koor))
print("Kelurahan ada di koordinat tapi tidak di pengaduan:", sorted(kel_koor - kel_raw))

# kecamatan di file koordinat vs kecamatan mayoritas di pengaduan (abaikan spasi)
maj = (allr.dropna(subset=["kel_norm", "kec_k"]).groupby(["kel_norm", "kec_k"]).size().reset_index(name="n")
       .sort_values("n", ascending=False).drop_duplicates("kel_norm").set_index("kel_norm")["kec_k"])
c["kec_k"] = c["kecamatan"].map(kec_key)
beda = c[c["kel_norm"].map(maj).notna() & (c["kel_norm"].map(maj) != c["kec_k"])]
print("\nKelurahan yang kecamatannya di koordinat != kecamatan mayoritas di pengaduan:", len(beda))
display(beda[["kelurahan", "kecamatan"]].assign(kecamatan_mayoritas_pengaduan=beda["kel_norm"].map(maj)))

## 11. Rekap lintas tahun

In [ ]:
overview = (allr.groupby("Tahun_file").agg(
    jumlah_pengaduan=("No Ticket", "size"),
    kelurahan_unik=("kel_norm", "nunique"),
    kecamatan_unik=("kec_k", "nunique"),
    kelurahan_missing=("kel_norm", lambda s: int(s.isna().sum())),
    keluhan_missing=("Keluhan", lambda s: int(s.isna().sum())),
).reset_index())
display(overview)

## 12. Tabel temuan → dasar aturan cleaning
Setiap aturan di notebook 02 harus bisa ditelusuri ke satu baris tabel ini.

In [ ]:
if n_kel_missing:
    top_kec_missing = tab_mk.drop(index="All").sort_values("All", ascending=False)
    share_top = float(top_kec_missing["All"].iloc[0] / n_kel_missing * 100)
    nama_top = top_kec_missing.index[0]
else:
    share_top, nama_top = 0.0, "-"

temuan = [
    ("Jumlah pengaduan mentah", len(allr), "Basis untuk tabel raw -> clean."),
    ("Duplikat baris utuh", n_dup_baris, "Tidak ada dedup jika 0."),
    ("Duplikat tiket (dalam tahun / antar tahun)", f"{n_dup_tiket_dalam} / {n_dup_tiket_antar}", "Dedup hanya bila > 0; tetap dicatat."),
    ("Tanggal invalid", n_tgl_invalid, "Baris tanggal invalid dikeluarkan (bila ada)."),
    ("Baris dgn tahun tanggal != tahun file", n_tahun_beda, "Tahun analisis diturunkan dari Tanggal Keluhan."),
    ("Kelurahan missing", f"{n_kel_missing} ({n_kel_missing/len(allr)*100:.2f}%)", "Tidak diimputasi; dikeluarkan dari analisis level kelurahan."),
    (f"Porsi missing Kelurahan pada kecamatan terbanyak ({nama_top})", f"{share_top:.1f}%", "Missing tidak acak -> catat sebagai keterbatasan."),
    ("Keluhan missing / kosong", f"{n_keluhan_missing} / {n_keluhan_kosong}", "Tidak bisa di-embedding -> dikeluarkan."),
    ("Kecamatan di luar 31 resmi", len(kec_tidak_resmi), "Alias hanya bila terbukti variasi ejaan."),
    ("Kelurahan dgn >1 Kecamatan", ambigu["Kelurahan"].nunique(), "Kunci pencocokan = nama Kelurahan (unik); Kecamatan mengikuti master."),
    ("Baris pada pasangan Kel-Kec minoritas", n_pair_minoritas, "Salah input Kecamatan -> dikoreksi & diaudit."),
    ("Teks <=10 karakter", n_pendek, "Tidak dibuang berdasar panjang (banyak yang valid)."),
    ("Teks >=2000 karakter", n_panjang, "Truncation ditangani di tahap embedding."),
    ("Baris teks identik (kelurahan & tahun sama)", n_dup_teks_lok, "Diberi flag, tidak dihapus otomatis."),
    ("Baris memuat nomor telepon", n_telepon, "Diberi flag; samarkan saat dikutip."),
    ("Topik unik", allr["Topik"].nunique(), "Perlu pengelompokan ke 16 kelompok topik."),
    ("Koordinat: baris / kelurahan unik", f"{coord_checks['jumlah_baris']} / {coord_checks['kelurahan_unik']}", f"Harus = {N_KELURAHAN}."),
    ("Koordinat: di luar bbox / duplikat lat-lon", f"{coord_checks['koordinat_di_luar_bbox_surabaya']} / {coord_checks['pasangan_lat_lon_duplikat']}", "Harus 0 / 0."),
    ("Koordinat: baris fallback", coord_checks.get("baris_fallback", "n/a"), "Harus 0 (koordinat kelurahan, bukan centroid kecamatan)."),
]
temuan = pd.DataFrame(temuan, columns=["temuan", "nilai", "implikasi_untuk_cleaning"])
display(temuan)

OUTPUT_DIR = BASE_DIR / "discovery_output"; OUTPUT_DIR.mkdir(exist_ok=True)
temuan.to_csv(OUTPUT_DIR / "discovery_findings.csv", index=False, encoding="utf-8-sig")
overview.to_csv(OUTPUT_DIR / "discovery_overview.csv", index=False)
tab_mk.to_csv(OUTPUT_DIR / "missing_kelurahan_per_kecamatan.csv")
ambigu.to_csv(OUTPUT_DIR / "kelurahan_multi_kecamatan.csv", index=False)
print("Tersimpan di:", OUTPUT_DIR)